# Assignment 6: Mixed-Precision Training and Efficiency (100 points)

This assignment covers techniques for training models efficiently: mixed-precision (FP16/BF16), gradient accumulation, memory optimization, and profiling.

## Background

Modern AI competition and research requires training large models under tight resource constraints. Key techniques:

- **Mixed-precision training:** Use FP16 or BF16 for most operations, FP32 for critical accumulations
- **Gradient accumulation:** Simulate large batch sizes with limited GPU memory
- **Gradient checkpointing:** Trade compute for memory by recomputing activations
- **Memory-efficient attention:** Reduce the quadratic memory cost of attention

### FP16 vs FP32 vs BF16

| Format | Exponent bits | Mantissa bits | Range | Precision |
|--------|:---:|:---:|---|---|
| FP32 | 8 | 23 | $\pm 3.4 \times 10^{38}$ | ~7 decimal digits |
| FP16 | 5 | 10 | $\pm 65504$ | ~3 decimal digits |
| BF16 | 8 | 7 | $\pm 3.4 \times 10^{38}$ | ~2 decimal digits |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.amp import autocast, GradScaler
import time
import math

---

> **WARNING:** Do not modify any code outside of the designated solution areas.

---

## Part 1: Understanding Number Formats (10 points)

**[Non-coding]**

1. (3 points) FP16 can represent numbers up to 65504. What happens if a gradient value exceeds this range during training? What is this problem called?

2. (3 points) BF16 has the same exponent range as FP32 but lower precision. Why does this make it more suitable for deep learning than FP16 in some cases?

3. (2 points) Why are loss values and gradient accumulations typically kept in FP32 even when the forward pass uses FP16?

4. (2 points) What is a "loss scaler" and why is it needed for FP16 but not BF16 training?

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

## Part 2: Mixed-Precision Forward Pass (12 points)

**[Coding]** Implement a simple model and demonstrate mixed-precision forward pass.

1. (4 points) Create a model: Linear(784, 256) → ReLU → Linear(256, 128) → ReLU → Linear(128, 10)

2. (4 points) Run a forward pass with FP32 input. Print the dtype of the output and intermediate activations.

3. (4 points) Run the same forward pass inside `torch.amp.autocast('cpu')` (or `'cuda'` if available). Print the dtype of the output. Observe which layers run in reduced precision.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

# Model definition and mixed-precision comparison

""" END OF THIS PART """

## Part 3: Mixed-Precision Training Loop (15 points)

**[Coding]** Implement a complete mixed-precision training loop using `torch.amp.autocast` and `GradScaler`.

- Use the model from Part 2
- Generate synthetic data: 1000 random samples of shape (784,), 10 classes
- Train for 10 epochs with batch_size=64, Adam lr=1e-3
- Use `autocast` for the forward pass and loss computation
- Use `GradScaler` for gradient scaling

**The training loop structure with AMP:**
```python
scaler = GradScaler()
for batch in loader:
    optimizer.zero_grad()
    with autocast(device_type):
        output = model(x)
        loss = criterion(output, y)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
```

In [ ]:
### WRITE YOUR SOLUTION HERE ###

# Mixed-precision training loop

""" END OF THIS PART """

## Part 4: Gradient Accumulation (15 points)

**[Coding]** Implement gradient accumulation to simulate a larger batch size.

If your GPU can only fit batch_size=16 but you want an effective batch_size=64, accumulate gradients over 4 mini-batches before stepping.

1. (10 points) Implement a training loop with gradient accumulation:
   - Physical batch size: 16
   - Accumulation steps: 4
   - Effective batch size: 64
   - **Important:** Scale the loss by `1/accumulation_steps` so the total gradient is averaged correctly

2. (5 points) Train both with and without accumulation (same effective batch size) for 5 epochs. Compare the final losses — they should be similar.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def train_with_accumulation(model, loader, optimizer, criterion,
                            accumulation_steps=4, epochs=5):
    """
    Train with gradient accumulation.
    """
    pass  # YOUR CODE

""" END OF THIS PART """

## Part 5: Gradient Checkpointing (12 points)

**[Coding]** Implement gradient checkpointing using `torch.utils.checkpoint.checkpoint`.

Gradient checkpointing trades compute for memory: instead of storing all intermediate activations for the backward pass, it recomputes them during backward.

1. (6 points) Create a deep model: 10 layers of (Linear(256, 256) → ReLU). Apply checkpointing to each layer.

2. (6 points) Compare memory usage (estimated by counting stored tensors) with and without checkpointing. Print the difference.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

from torch.utils.checkpoint import checkpoint

class CheckpointedModel(nn.Module):
    def __init__(self, d=256, num_layers=10):
        super().__init__()
        pass  # YOUR CODE

    def forward(self, x, use_checkpoint=False):
        pass  # YOUR CODE

""" END OF THIS PART """

## Part 6: Memory-Efficient Attention (12 points)

**[Coding]** Standard attention materializes the $(L, L)$ attention matrix. Implement a memory-efficient version.

1. (6 points) Implement standard attention that creates the full $(B, h, L, L)$ attention matrix. Estimate its memory usage for $B=1, h=8, L=4096, d_k=64$.

2. (6 points) Use `F.scaled_dot_product_attention` (PyTorch 2.0+) which uses Flash Attention under the hood. Compare the memory usage and speed.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def standard_attention(Q, K, V):
    """Standard attention with explicit attention matrix."""
    pass  # YOUR CODE

def efficient_attention(Q, K, V):
    """Memory-efficient attention using F.scaled_dot_product_attention."""
    pass  # YOUR CODE

# Compare

""" END OF THIS PART """

## Part 7: Combined Efficiency Techniques (14 points)

**[Coding]** Build a training pipeline that combines ALL efficiency techniques:

- Mixed-precision (autocast + GradScaler)
- Gradient accumulation (4 steps)
- A transformer-like model with 4 attention layers

1. (8 points) Implement the combined training loop.
2. (6 points) Compare training speed (time per epoch) with and without the efficiency techniques.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

# Combined efficiency training pipeline

""" END OF THIS PART """

## Part 8: Analysis (10 points)

**[Non-coding]**

1. (3 points) In a competition setting with limited GPU memory, rank the following techniques by their impact on enabling larger models: mixed-precision, gradient accumulation, gradient checkpointing. Justify your ranking.

2. (3 points) You are training a PINN and want to use mixed-precision. The PDE residual involves second-order derivatives computed via `autograd.grad`. What potential issues might arise with FP16 precision in the autograd computation?

3. (4 points) Design an experiment to determine the optimal combination of batch_size and accumulation_steps for a given GPU memory budget. What metric would you optimize?

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """